In [11]:
pip install charset_normalizer

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
pip install regex

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from pathlib import Path
from charset_normalizer import from_path
import re

1. Write a robust file reader function

In [14]:
def file_reader() -> tuple[str, str]: 
    """This function reads the files, decodes the files and shows which decoding was used."""

    # Get all files from the nested folders
    BASE_DIR = Path.cwd() / "data"
    files = [p for p in (BASE_DIR / "2026").rglob("*") if p.is_file()]

    # For each file that was found, decode and give type of encoding used
    for f in files:
        # Verify the files were found correctly
        print(f)

        # Read the raw bytes once, so the file is only opened a single time
        raw = f.read_bytes()

        # Try UTF-8
        # Invalid sequences raise an error.
        try:
            return raw.decode("utf-8-sig"), "utf-8"     # "utf-8-sig" strips BOM if present
        except UnicodeDecodeError:
            pass

        # Else decode it as ascii
        return raw.decode("iso-8859-1"), "ascii"

In [15]:
text, encoding = file_reader()
print(encoding)
print(text)

c:\Users\kikib\OneDrive\Documenten\COURSES\INF201-AdvancedProgramming\inf201_deliverable_2\data\2026\student_union\POS_01\register_sales.log
utf-8
[2026-10-15 07:02:52] [POS_01] item=Cappuccino price=4.49 NOK (CANCELLED)
[2026-10-15 07:03:14] [POS_01] item=Croissant price=4.26 USD (SUCCESS)
[2026-10-15 07:04:39] [POS_01] item=Muffin price=3.21 NOK (SUCCESS)
[2026-10-15 07:05:29] [POS_01] item=Cappuccino price=4.13 NOK (SUCCESS)
=== NETWORK SYNC OK ===
[2026-10-15 07:07:40] [POS_01] item=Latte price=4.34 NOK (CANCELLED)
[2026-10-15 07:09:14] [POS_01] item=Cappuccino price=4.50 NOK (SUCCESS)
[2026-10-15 07:11:16] [POS_01] item=Muffin price=3.87 USD (SUCCESS)
--- PRINTER PAPER LOW ---
[2026-10-15 07:13:52] [POS_01] item=Latte price=4.24 NOK (SUCCESS)
=== NETWORK SYNC OK ===
[2026-10-15 07:15:15] [POS_01] item=Latte price=4.36 EUR (SUCCESS)
--- PRINTER PAPER LOW ---
[2026-10-15 07:17:48] [POS_01] item=Croissant price=3.66 EUR (REFUND)
--- PRINTER PAPER LOW ---
[2026-10-15 07:20:15] [POS_01

2. Write a log parser function that uses regular expressions

In [16]:

def parse_log(text: str) -> list[dict]:
    """This function extracts valid sales records from log text, ignoring noise lines."""

    # Define what a valid line should look like
    VALID_LINE = re.compile(
        r"""
        ^\[(?P<timestamp>\d{4}-\d{2}-\d{2}\ \d{2}:\d{2}:\d{2})\]   # [2026-10-15 08:30:00]
        \s+\[(?P<register_id>[A-Za-z0-9_]+)\]                      # [POS_01]
        \s+item=(?P<item>.+?)                                      # Latte (spaces allowed)
        \s+price=(?P<price>\d+(?:\.\d+)?)                          # 4.50
        \s+(?P<currency>[A-Z]{3})                                  # EUR, NOK, USD
        \s+\((?P<status>SUCCESS|REFUND|CANCELLED)\)\s*$            # (SUCCESS)
        """,
        re.VERBOSE,
    )

    # Define list to store the valid sales lines 
    valid_sales_lines = []

    # Iterate over each line, store matching lines and remove the noise
    for line in text.splitlines():
        match = VALID_LINE.match(line.strip())

        if not match:
            continue  # skip the noise

        # Store a dictionary of all the named groups from the matches
        record = match.groupdict()

        # Convert the price into a floa
        record["price"] = float(record["price"])

        # Add record to valid_sales_lines list
        valid_sales_lines.append(record)

    return valid_sales_lines

In [17]:
valid_sales = parse_log(text)

valid_sales

[{'timestamp': '2026-10-15 07:02:52',
  'register_id': 'POS_01',
  'item': 'Cappuccino',
  'price': 4.49,
  'currency': 'NOK',
  'status': 'CANCELLED'},
 {'timestamp': '2026-10-15 07:03:14',
  'register_id': 'POS_01',
  'item': 'Croissant',
  'price': 4.26,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:04:39',
  'register_id': 'POS_01',
  'item': 'Muffin',
  'price': 3.21,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:05:29',
  'register_id': 'POS_01',
  'item': 'Cappuccino',
  'price': 4.13,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:07:40',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.34,
  'currency': 'NOK',
  'status': 'CANCELLED'},
 {'timestamp': '2026-10-15 07:09:14',
  'register_id': 'POS_01',
  'item': 'Cappuccino',
  'price': 4.5,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:11:16',
  'register_id': 'POS_01',
  'item': 'Muffin',
  'price': 3.87,